<a href="https://colab.research.google.com/github/SakaSaheed/AI-ML-Project/blob/main/Smart_Exam_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Smart Exam Hall Tracker using RFID and SDcard to log the activities.

Note: Disconnect the MISO trace on SDcard and connect it directly to pin 3 (from right) on SDCard pinout. This will allow SDcard and RFID to work together.


In [ ]:
//Working code

#include <SPI.h>
#include <MFRC522.h>
#include <LiquidCrystal_I2C.h>
#include <SD.h>
#include <Wire.h>
#include <RTClib.h>

// ========== PIN DEFINITIONS ==========
#define SD_CS_PIN   2     // SD Card Chip Select
#define RST_PIN     0     // RFID RST - D3
#define SS_PIN      15    // RFID SDA - D8
#define BUZZER_PIN  16    // Buzzer - D0
#define RESET_BTN   3     // Reset button - RX

// LCD (I2C - address 0x27 or 0x3F)
LiquidCrystal_I2C lcd(0x27, 16, 2);

MFRC522 mfrc522(SS_PIN, RST_PIN);
RTC_DS3231 rtc;

// ========== DYNAMIC STUDENT DATABASE (No Pre-programmed) ==========
struct StudentRecord {
  String uid;
  String name;
  int uniqueID;
  bool registered;
};

StudentRecord registeredStudents[50];
int registeredCount = 0;
int nextUniqueID = 1001;  // Start from 1001

// ========== ACTIVE SESSION TRACKING ==========
struct ActiveStudent {
  String uid;
  String name;
  int uniqueID;
  String status;        // "REGISTERED", "IN_HALL", "OUT_TOILET"
  DateTime entryTime;
  DateTime exitTime;
  DateTime lastToiletExit;
  int toiletBreakCount;
  bool violated;
  unsigned long violationAlertTime;
};

ActiveStudent activeStudents[50];
int activeCount = 0;

// ========== EXAM TIMING ==========
unsigned long powerOnTime = 0;
bool examStarted = false;
bool examEnded = false;
bool registrationOpen = true;
unsigned long examStartTime = 0;
unsigned long examEndTime = 0;

const unsigned long REGISTRATION_WINDOW = 60000;   // 1 minute for registration
const unsigned long EXAM_DURATION = 360000;        // 6 minutes total
const unsigned long TOILET_BLOCK_TIME = 60000;     // No toilet breaks for 1 min after exam start
const unsigned long TOILET_LIMIT = 30000;          // 30 seconds max for toilet break

// System state
bool buzzerActive = false;
unsigned long buzzerStartTime = 0;
unsigned long lastLCDUpdate = 0;
String lastViolatorName = "";
unsigned long violationDisplayTime = 0;
bool sdCardOK = false;

// File for logging
File myFile;
long logEntryID = 1;
String currentLogFileName = "EXAM_LOG.csv";

void setup() {
  Serial.begin(115200);
  Wire.begin();
  SPI.begin();
  powerOnTime = millis();

  // Initialize LCD
  lcd.init();
  lcd.backlight();
  lcd.clear();
  lcd.setCursor(0, 0);
  lcd.print("Exam Control Sys");
  lcd.setCursor(0, 1);
  lcd.print("Power ON");
  delay(2000);

  // Initialize components
  lcd.setCursor(0, 1);
  lcd.print("Init RFID...");
  mfrc522.PCD_Init();
  delay(500);

  lcd.setCursor(0, 1);
  lcd.print("Init RTC...");
  initRTC();
  delay(500);

  lcd.setCursor(0, 1);
  lcd.print("Init SD Card...");
  initSDCard();
  delay(500);

  // Setup pins
  pinMode(BUZZER_PIN, OUTPUT);
  pinMode(RESET_BTN, INPUT_PULLUP);
  digitalWrite(BUZZER_PIN, LOW);

  // Create log file header
  createLogFile();

  // Calculate exam times
  examStartTime = powerOnTime + REGISTRATION_WINDOW;
  examEndTime = powerOnTime + REGISTRATION_WINDOW + EXAM_DURATION;

  // Print system info
  Serial.println("========================================");
  Serial.println("   DYNAMIC REGISTRATION EXAM SYSTEM");
  Serial.println("========================================");
  Serial.print("Power ON at: ");
  Serial.println(millis() / 1000);
  Serial.print("Registration Window: 1 minute (until ");
  Serial.print(examStartTime / 1000);
  Serial.println(" seconds)");
  Serial.print("Exam Duration: 6 minutes (ends at ");
  Serial.print(examEndTime / 1000);
  Serial.println(" seconds)");
  Serial.print("Toilet Break Limit: 30 seconds");
  Serial.println("Register your card during registration window!");
  Serial.println("========================================");

  lcd.clear();
  lcd.print("REGISTRATION");
  lcd.setCursor(0, 1);
  lcd.print("Scan your card");
  delay(2000);

  updateRegistrationDisplay();
}

void initSDCard() {
  pinMode(SD_CS_PIN, OUTPUT);

  if (!SD.begin(SD_CS_PIN)) {
    Serial.println("SD Card initialization failed!");
    sdCardOK = false;
  } else {
    Serial.println("SD Card initialized successfully");
    sdCardOK = true;
  }
}

void createLogFile() {
  if (!sdCardOK) return;

  if (!SD.exists(currentLogFileName)) {
    File dataFile = SD.open(currentLogFileName, FILE_WRITE);
    if (dataFile) {
      dataFile.println("ID,Timestamp,Student Name,UID,UniqueID,Event,Duration,Status");
      dataFile.close();
      Serial.println("Log file created: " + currentLogFileName);
    }
  }
}

void writeToSDCard(String dataString) {
  if (!sdCardOK) return;

  myFile = SD.open(currentLogFileName, FILE_WRITE);
  if (myFile) {
    myFile.println(dataString);
    myFile.close();
    Serial.println("SD: " + dataString);
  }
}

void logEvent(String studentName, String uid, int uniqueID, String event, String duration, String status) {
  if (!sdCardOK) return;

  DateTime now = rtc.now();
  String timestamp = getDateTimeString(now);

  String dataString = String(logEntryID) + ",";
  dataString += timestamp + ",";
  dataString += studentName + ",";
  dataString += uid + ",";
  dataString += String(uniqueID) + ",";
  dataString += event + ",";
  dataString += duration + ",";
  dataString += status;

  writeToSDCard(dataString);
  logEntryID++;
}

void initRTC() {
  if (!rtc.begin()) {
    Serial.println("RTC not found!");
  } else {
    Serial.println("RTC ready");
  }
}

String getDateTimeString(DateTime dt) {
  char buffer[25];
  sprintf(buffer, "%04d-%02d-%02d %02d:%02d:%02d",
          dt.year(), dt.month(), dt.day(),
          dt.hour(), dt.minute(), dt.second());
  return String(buffer);
}

String getTimeString(DateTime dt) {
  char buffer[9];
  sprintf(buffer, "%02d:%02d:%02d",
          dt.hour(), dt.minute(), dt.second());
  return String(buffer);
}

String readUID() {
  String uid = "";
  for (byte i = 0; i < mfrc522.uid.size; i++) {
    if (mfrc522.uid.uidByte[i] < 0x10) uid += "0";
    uid += String(mfrc522.uid.uidByte[i], HEX);
  }
  return uid;
}

// Check if UID is already registered
bool isUIDRegistered(String uid) {
  for (int i = 0; i < registeredCount; i++) {
    if (registeredStudents[i].uid == uid) {
      return true;
    }
  }
  return false;
}

// Get student name from registration (prompts user to enter name)
String getStudentNameFromRegistration(String uid) {
  // For dynamic registration, we'll use a generic name
  // The supervisor can later map UID to real names from the log file
  return "Student_" + String(nextUniqueID);
}

// Get student info by UID
int findRegisteredStudent(String uid) {
  for (int i = 0; i < registeredCount; i++) {
    if (registeredStudents[i].uid == uid) {
      return i;
    }
  }
  return -1;
}

int findActiveStudent(String uid) {
  for (int i = 0; i < activeCount; i++) {
    if (activeStudents[i].uid == uid) return i;
  }
  return -1;
}

// Register a new student during registration window
void registerStudent(String uid) {
  if (registeredCount >= 50) {
    lcd.clear();
    lcd.print("REGISTRATION");
    lcd.setCursor(0, 1);
    lcd.print("CLASS FULL!");
    Serial.println("Registration failed: Class is full!");
    delay(2000);
    return;
  }

  int uniqueID = nextUniqueID++;
  String studentName = "Student_" + String(uniqueID);

  // Store in registered students database
  registeredStudents[registeredCount].uid = uid;
  registeredStudents[registeredCount].name = studentName;
  registeredStudents[registeredCount].uniqueID = uniqueID;
  registeredStudents[registeredCount].registered = true;
  registeredCount++;

  // Also add to active session
  activeStudents[activeCount].uid = uid;
  activeStudents[activeCount].name = studentName;
  activeStudents[activeCount].uniqueID = uniqueID;
  activeStudents[activeCount].status = "REGISTERED";
  activeStudents[activeCount].entryTime = rtc.now();
  activeStudents[activeCount].toiletBreakCount = 0;
  activeStudents[activeCount].violated = false;
  activeCount++;

  // Display success
  lcd.clear();
  lcd.print("REGISTERED!");
  lcd.setCursor(0, 1);
  lcd.print("ID: ");
  lcd.print(uniqueID);

  Serial.print("NEW REGISTRATION: ");
  Serial.print(studentName);
  Serial.print(" | UID: ");
  Serial.print(uid);
  Serial.print(" | Unique ID: ");
  Serial.println(uniqueID);

  logEvent(studentName, uid, uniqueID, "REGISTERED", "N/A", "SUCCESS");

  delay(1500);
  updateRegistrationDisplay();
}

void updateRegistrationDisplay() {
  unsigned long remaining = 0;
  if (examStartTime > millis()) {
    remaining = (examStartTime - millis()) / 1000;
  }

  lcd.clear();
  lcd.print("REGISTRATION");
  lcd.setCursor(0, 1);
  lcd.print("Time: ");
  lcd.print(remaining);
  lcd.print("s  Reg: ");
  lcd.print(registeredCount);
  lcd.print("/50");
}

void updateExamDisplay() {
  unsigned long remaining = (examEndTime - millis()) / 1000;

  lcd.clear();
  lcd.print("EXAM IN PROGRESS");
  lcd.setCursor(0, 1);
  lcd.print("Time: ");
  lcd.print(remaining / 60);
  lcd.print(":");
  if ((remaining % 60) < 10) lcd.print("0");
  lcd.print(remaining % 60);
  lcd.print("  Hall: ");
  lcd.print(activeCount);
}

void handleScan(String uid) {
  unsigned long currentTime = millis();
  DateTime now = rtc.now();

  // Case 1: Registration Phase (First 1 minute)
  if (currentTime < examStartTime) {
    if (!isUIDRegistered(uid)) {
      // New student - register them
      registerStudent(uid);
    } else {
      // Already registered
      int regIndex = findRegisteredStudent(uid);
      if (regIndex >= 0) {
        lcd.clear();
        lcd.print("ALREADY REG'D");
        lcd.setCursor(0, 1);
        lcd.print("ID: ");
        lcd.print(registeredStudents[regIndex].uniqueID);
        Serial.print("Already registered: ");
        Serial.println(registeredStudents[regIndex].name);
        delay(1000);
        updateRegistrationDisplay();
      }
    }
    return;
  }

  // Case 2: Exam Phase
  if (currentTime >= examStartTime && currentTime < examEndTime) {
    // Check if student is registered
    if (!isUIDRegistered(uid)) {
      lcd.clear();
      lcd.print("NOT REGISTERED!");
      lcd.setCursor(0, 1);
      lcd.print("Regd. closed");
      Serial.print("REJECTED: Unregistered UID ");
      Serial.println(uid);
      logEvent("UNKNOWN", uid, 0, "REJECTED_ENTRY", "N/A", "NOT_REGISTERED");
      delay(2000);
      updateExamDisplay();
      return;
    }

    // Get registered student info
    int regIndex = findRegisteredStudent(uid);
    if (regIndex < 0) return;

    String studentName = registeredStudents[regIndex].name;
    int uniqueID = registeredStudents[regIndex].uniqueID;
    int activeIndex = findActiveStudent(uid);

    // If exam just started, move all registered students to IN_HALL
    if (!examStarted) {
      examStarted = true;
      for (int i = 0; i < activeCount; i++) {
        if (activeStudents[i].status == "REGISTERED") {
          activeStudents[i].status = "IN_HALL";
          logEvent(activeStudents[i].name, activeStudents[i].uid, activeStudents[i].uniqueID,
                   "EXAM_STARTED", "N/A", "IN_HALL");
        }
      }
      lcd.clear();
      lcd.print("EXAM STARTED!");
      lcd.setCursor(0, 1);
      lcd.print("Good Luck!");
      delay(2000);
      updateExamDisplay();
    }

    // Handle toilet breaks
    if (activeStudents[activeIndex].status == "IN_HALL") {
      // Check if toilet breaks are allowed (after 1 minute of exam)
      if (currentTime - examStartTime < TOILET_BLOCK_TIME) {
        lcd.clear();
        lcd.print("TOILET BREAKS");
        lcd.setCursor(0, 1);
        lcd.print("Not yet allowed");
        Serial.print("TOILET DENIED: ");
        Serial.print(studentName);
        Serial.println(" - First minute of exam");
        delay(2000);
        updateExamDisplay();
        return;
      }

      // Allow toilet exit
      activeStudents[activeIndex].status = "OUT_TOILET";
      activeStudents[activeIndex].lastToiletExit = now;
      activeStudents[activeIndex].toiletBreakCount++;

      lcd.clear();
      lcd.print("TOILET BREAK");
      lcd.setCursor(0, 1);
      lcd.print("30 sec limit");
      Serial.print("TOILET EXIT: ");
      Serial.print(studentName);
      Serial.print(" (ID:");
      Serial.print(uniqueID);
      Serial.print(" Break #");
      Serial.print(activeStudents[activeIndex].toiletBreakCount);
      Serial.println(")");

      logEvent(studentName, uid, uniqueID, "TOILET_EXIT",
               String(activeStudents[activeIndex].toiletBreakCount), "TIMER_STARTED");
      delay(1500);
      updateExamDisplay();
    }
    else if (activeStudents[activeIndex].status == "OUT_TOILET") {
      // Returning from toilet
      long absenceSeconds = calculateAbsenceSeconds(activeStudents[activeIndex].lastToiletExit, now);
      unsigned long absenceMs = absenceSeconds * 1000;

      if (absenceMs > TOILET_LIMIT) {
        // VIOLATION - exceeded 30 seconds
        String duration = String(absenceSeconds) + "s";
        logViolation(studentName, uid, uniqueID, absenceSeconds, now,
                    activeStudents[activeIndex].toiletBreakCount);
        activeStudents[activeIndex].violated = true;
        activeStudents[activeIndex].violationAlertTime = millis();

        lcd.clear();
        lcd.print("VIOLATION!");
        lcd.setCursor(0, 1);
        lcd.print("Exceeded 30s");

        buzzerActive = true;
        buzzerStartTime = millis();
        digitalWrite(BUZZER_PIN, HIGH);

        Serial.print("!!! VIOLATION !!! ");
        Serial.print(studentName);
        Serial.print(" (ID:");
        Serial.print(uniqueID);
        Serial.print(") - Toilet break #");
        Serial.print(activeStudents[activeIndex].toiletBreakCount);
        Serial.print(" lasted ");
        Serial.print(absenceSeconds);
        Serial.println(" seconds (exceeded 30s limit)");

        delay(3000);
      } else {
        // Returned on time
        lcd.clear();
        lcd.print("RETURNED");
        lcd.setCursor(0, 1);
        lcd.print("Time: ");
        lcd.print(absenceSeconds);
        lcd.print(" sec");

        Serial.print("RETURNED: ");
        Serial.print(studentName);
        Serial.print(" (ID:");
        Serial.print(uniqueID);
        Serial.print(") - Toilet break #");
        Serial.print(activeStudents[activeIndex].toiletBreakCount);
        Serial.print(" lasted ");
        Serial.print(absenceSeconds);
        Serial.println(" seconds");

        logEvent(studentName, uid, uniqueID, "TOILET_RETURN",
                 String(absenceSeconds) + "s", "ON_TIME");
        delay(1500);
      }

      activeStudents[activeIndex].status = "IN_HALL";
      updateExamDisplay();
    }
    return;
  }

  // Case 3: Exam Ended
  if (currentTime >= examEndTime && !examEnded) {
    examEnded = true;
    lcd.clear();
    lcd.print("EXAM ENDED");
    lcd.setCursor(0, 1);
    lcd.print("System Closing");
    Serial.println("=== EXAM ENDED ===");

    // Log all students who are still out
    for (int i = 0; i < activeCount; i++) {
      if (activeStudents[i].status == "OUT_TOILET") {
        logEvent(activeStudents[i].name, activeStudents[i].uid, activeStudents[i].uniqueID,
                 "STILL_OUT_AT_END", "N/A", "VIOLATION");
      }
    }
    delay(3000);
  }

  if (examEnded) {
    lcd.clear();
    lcd.print("EXAM FINISHED");
    lcd.setCursor(0, 1);
    lcd.print("See Examiner");
    return;
  }
}

long calculateAbsenceSeconds(DateTime exitTime, DateTime returnTime) {
  TimeSpan diff = returnTime - exitTime;
  return diff.totalseconds();
}

void logViolation(String name, String uid, int uniqueID, long absenceSeconds, DateTime violationTime, int breakCount) {
  String duration = String(absenceSeconds) + "s";

  lastViolatorName = name;
  violationDisplayTime = millis();

  Serial.println("======================================");
  Serial.print("!!! VIOLATION DETECTED !!!\n");
  Serial.print("Student: ");
  Serial.print(name);
  Serial.print(" (ID: ");
  Serial.print(uniqueID);
  Serial.print(")\nToilet Break #");
  Serial.print(breakCount);
  Serial.print("\nDuration: ");
  Serial.print(absenceSeconds);
  Serial.println(" seconds (Limit: 30 seconds)");
  Serial.println("======================================");

  if (sdCardOK) {
    String timestamp = getDateTimeString(violationTime);
    String dataString = String(logEntryID) + ",";
    dataString += timestamp + ",";
    dataString += name + ",";
    dataString += uid + ",";
    dataString += String(uniqueID) + ",";
    dataString += "VIOLATION!!!,";
    dataString += duration + ",";
    dataString += "EXCEEDED_30S_LIMIT";
    writeToSDCard(dataString);
    logEntryID++;
  }
}

void updateLCD() {
  if (millis() - lastLCDUpdate > 200) {
    lastLCDUpdate = millis();

    if (violationDisplayTime > 0 && (millis() - violationDisplayTime < 5000)) {
      lcd.clear();
      lcd.print("VIOLATOR!");
      lcd.setCursor(0, 1);
      lcd.print(lastViolatorName);
      return;
    }

    // Show countdown for students on toilet break
    for (int i = 0; i < activeCount; i++) {
      if (activeStudents[i].status == "OUT_TOILET" && !activeStudents[i].violated) {
        DateTime now = rtc.now();
        long absenceSeconds = calculateAbsenceSeconds(activeStudents[i].lastToiletExit, now);
        unsigned long absenceMs = absenceSeconds * 1000;

        if (absenceMs <= TOILET_LIMIT) {
          unsigned long remaining = (TOILET_LIMIT - absenceMs) / 1000;

          lcd.clear();
          String studentName = activeStudents[i].name;
          if (studentName.length() > 8) {
            lcd.print(studentName.substring(0, 8));
          } else {
            lcd.print(studentName);
          }
          lcd.setCursor(0, 1);
          lcd.print("Return in: ");
          lcd.print(remaining);
          lcd.print("s");
          return;
        }
      }
    }
  }
}

void manageBuzzer() {
  if (buzzerActive && (millis() - buzzerStartTime > 5000)) {
    digitalWrite(BUZZER_PIN, LOW);
    buzzerActive = false;
  }
}

void handleResetButton() {
  digitalWrite(BUZZER_PIN, LOW);
  buzzerActive = false;

  DateTime now = rtc.now();

  if (sdCardOK) {
    String timestamp = getDateTimeString(now);
    String dataString = String(logEntryID) + ",";
    dataString += timestamp + ",SUPERVISOR,RESET_BUTTON,0,ALERT_RESET,0,RESET";
    writeToSDCard(dataString);
    logEntryID++;
  }

  lcd.clear();
  lcd.print("ALERT RESET");
  lcd.setCursor(0, 1);
  lcd.print("Supervisor OK");
  delay(1500);

  Serial.println("=== SYSTEM RESET by Supervisor ===");
}

void printDateTime(DateTime dt) {
  Serial.print(dt.year(), DEC);
  Serial.print('/');
  Serial.print(dt.month(), DEC);
  Serial.print('/');
  Serial.print(dt.day(), DEC);
  Serial.print(' ');
  Serial.print(dt.hour(), DEC);
  Serial.print(':');
  Serial.print(dt.minute(), DEC);
  Serial.print(':');
  Serial.print(dt.second(), DEC);
}

void loop() {
  unsigned long currentTime = millis();

  // Update display based on phase
  if (currentTime < examStartTime) {
    static unsigned long lastRegUpdate = 0;
    if (currentTime - lastRegUpdate > 1000) {
      lastRegUpdate = currentTime;
      updateRegistrationDisplay();
    }
  } else if (currentTime >= examStartTime && currentTime < examEndTime) {
    static unsigned long lastExamUpdate = 0;
    if (currentTime - lastExamUpdate > 1000) {
      lastExamUpdate = currentTime;
      if (!examStarted) {
        // Trigger exam start with dummy scan
        handleScan("dummy");
      } else {
        updateExamDisplay();
      }
    }
  }

  // Check reset button
  if (digitalRead(RESET_BTN) == LOW) {
    handleResetButton();
    delay(500);
  }

  // Check for RFID scans
  if (mfrc522.PICC_IsNewCardPresent() && mfrc522.PICC_ReadCardSerial()) {
    String uid = readUID();
    handleScan(uid);

    mfrc522.PICC_HaltA();
    delay(500);
  }

  updateLCD();
  manageBuzzer();
}